# 02b — Distributed fine-tuning with Ray Train + TRL

**Purpose:** The scale-out variant of `02a`. Runs the **same** TRL LoRA fine-tuning, but distributed across multiple GPU workers/nodes using **Ray Train** (`TorchTrainer`), reading the ChatML **Delta table** from `01` via Ray Data. This matches the Ray stack used in the `multimodal-lance` blueprint.

Same dataset, same `BASE_MODEL`, same LoRA config as `02a` — only the execution engine differs.

> **Cluster:** a Ray-on-Spark GPU cluster (multi-node). Set `NUM_WORKERS` to your total GPU count.

In [ ]:
%pip install -q -U "ray[train]" transformers peft trl datasets accelerate mlflow
dbutils.library.restartPython()

In [ ]:
# ─────────────────────────────────────────────────────────────
# CONFIGURATION  (shared with 02a)
# ─────────────────────────────────────────────────────────────
BASE_MODEL = "meta-llama/Llama-3.2-1B-Instruct"

CATALOG = "main"
SCHEMA  = "otel_finetuning"
CHATML_TABLE = f"{CATALOG}.{SCHEMA}.chatml_dataset"
VOLUME       = f"/Volumes/{CATALOG}/{SCHEMA}/finetune"
OUTPUT_DIR   = f"{VOLUME}/lora_adapter_ray"

NUM_WORKERS = 2        # total training workers (= number of GPUs)
USE_GPU     = True

LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]
EPOCHS, BATCH_SIZE, LR, MAX_SEQ_LEN = 2, 4, 2e-4, 1024

MLFLOW_EXPERIMENT = "/Shared/otel-finetuning"

## Start Ray on the Databricks cluster

`setup_ray_cluster` bootstraps a Ray cluster on top of the Spark workers.

In [ ]:
from ray.util.spark import setup_ray_cluster, shutdown_ray_cluster
import ray

# Idempotent: shut down any prior cluster before starting a new one.
try:
    shutdown_ray_cluster()
except Exception:
    pass

setup_ray_cluster(max_worker_nodes=NUM_WORKERS, num_gpus_worker_node=1)
ray.init(ignore_reinit_error=True)
print(ray.cluster_resources())

## Load the ChatML Delta table as a Ray Dataset

In [ ]:
# Ray reads the Delta table written by 01. Each row has messages_json + split.
train_ds = ray.data.read_databricks_tables(  # or ray.data.read_delta on the table path
    warehouse_id=None, catalog=CATALOG, schema=SCHEMA, query=f"SELECT messages_json FROM {CHATML_TABLE} WHERE split='train'"
) if False else None

# Simpler + portable: pull rows via Spark, hand them to Ray as an in-memory dataset.
import json
rows = spark.table(CHATML_TABLE).where("split='train'").select("messages_json").collect()
train_records = [{"messages": json.loads(r["messages_json"])} for r in rows]
print(f"{len(train_records)} training examples")

## Define the per-worker training function

In [ ]:
def train_func(config):
    """Runs on each Ray worker — identical TRL SFT logic to 02a."""
    import torch, ray.train
    from datasets import Dataset
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import LoraConfig
    from trl import SFTConfig, SFTTrainer

    # Ray shards the dataset; each worker gets its slice.
    shard = ray.train.get_dataset_shard("train")
    records = list(shard.iter_rows())
    ds = Dataset.from_list([{"messages": r["messages"]} for r in records])

    tok = AutoTokenizer.from_pretrained(config["base_model"])
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(config["base_model"], torch_dtype=torch.bfloat16)

    peft_config = LoraConfig(
        r=config["lora_r"], lora_alpha=config["lora_alpha"], lora_dropout=config["lora_dropout"],
        target_modules=config["target_modules"], task_type="CAUSAL_LM", bias="none")

    sft_config = SFTConfig(
        output_dir="/tmp/ray_sft", num_train_epochs=config["epochs"],
        per_device_train_batch_size=config["batch_size"], learning_rate=config["lr"],
        max_length=config["max_seq_len"], logging_steps=10, bf16=True, report_to=[])

    trainer = SFTTrainer(model=model, args=sft_config, train_dataset=ds,
                         peft_config=peft_config, processing_class=tok)
    trainer.train()

    # Rank-0 worker persists the adapter to the shared Volume.
    if ray.train.get_context().get_world_rank() == 0:
        trainer.save_model(config["output_dir"])

## Launch distributed training

In [ ]:
from ray.train.torch import TorchTrainer
from ray.train import ScalingConfig
import mlflow

train_config = {
    "base_model": BASE_MODEL, "lora_r": LORA_R, "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT, "target_modules": TARGET_MODULES,
    "epochs": EPOCHS, "batch_size": BATCH_SIZE, "lr": LR,
    "max_seq_len": MAX_SEQ_LEN, "output_dir": OUTPUT_DIR,
}

ray_ds = ray.data.from_items(train_records)

trainer = TorchTrainer(
    train_loop_per_worker=train_func,
    train_loop_config=train_config,
    datasets={"train": ray_ds},
    scaling_config=ScalingConfig(num_workers=NUM_WORKERS, use_gpu=USE_GPU),
)

mlflow.set_experiment(MLFLOW_EXPERIMENT)
with mlflow.start_run(run_name="lora-ray-train"):
    mlflow.log_params({"base_model": BASE_MODEL, "num_workers": NUM_WORKERS, "epochs": EPOCHS})
    result = trainer.fit()
    print("Done:", result.metrics)
    print("Adapter saved to", OUTPUT_DIR)

shutdown_ray_cluster()